# Predicting 30-Day Hospital Readmission

This notebook develops and evaluates machine-learning models for predicting readmission within 30 days.

To prevent patient-level data leakage, all encounters belonging to one patient are assigned to either the training set or the test set—never both. Preprocessing and model fitting are performed using training data only.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
project_root = Path.cwd()

if not (project_root / "data").exists():
    project_root = project_root.parent

clean_data_path = (
    project_root
    / "data"
    / "processed"
    / "hospital_readmissions_clean.csv"
)

tables_path = project_root / "reports" / "tables"
tables_path.mkdir(parents=True, exist_ok=True)

encounters = pd.read_csv(clean_data_path, low_memory=False)

print("Encounters:", f"{len(encounters):,}")
print("Unique patients:", f"{encounters['patient_nbr'].nunique():,}")
print(
    "Readmission rate:",
    f"{encounters['readmitted_30d'].mean() * 100:.2f}%"
)

Encounters: 99,340
Unique patients: 69,987
Readmission rate: 11.39%


## Feature exclusions

The following variables are excluded from the predictor matrix:

- `encounter_id`: Arbitrary encounter identifier
- `patient_nbr`: Patient identifier used only for group splitting
- `readmitted`: Original version of the outcome
- `readmitted_30d`: Binary model target
- `diag_1`, `diag_2`, and `diag_3`: Sparse raw ICD-9 codes replaced by broader diagnosis groups
- Numeric admission, discharge, and admission-source IDs: Replaced by readable categorical descriptions

All retained predictors describe information available by the time of discharge. No feature directly contains the subsequent readmission outcome.

In [3]:
excluded_features = [
    "encounter_id",
    "patient_nbr",
    "readmitted",
    "readmitted_30d",
    "diag_1",
    "diag_2",
    "diag_3",
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id"
]

feature_columns = [
    column
    for column in encounters.columns
    if column not in excluded_features
]

X = encounters[feature_columns].copy()
y = encounters["readmitted_30d"].copy()
groups = encounters["patient_nbr"].copy()

print("Predictor columns:", X.shape[1])
print("Target observations:", len(y))

Predictor columns: 44
Target observations: 99340


In [4]:
group_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_index, test_index = next(
    group_split.split(X, y, groups=groups)
)

X_train = X.iloc[train_index].copy()
X_test = X.iloc[test_index].copy()

y_train = y.iloc[train_index].copy()
y_test = y.iloc[test_index].copy()

groups_train = groups.iloc[train_index].copy()
groups_test = groups.iloc[test_index].copy()

In [5]:
shared_patients = set(groups_train).intersection(set(groups_test))

assert len(shared_patients) == 0
assert len(X_train) + len(X_test) == len(X)
assert X_train.index.equals(y_train.index)
assert X_test.index.equals(y_test.index)

split_summary = pd.DataFrame({
    "split": ["Training", "Testing"],
    "encounters": [len(X_train), len(X_test)],
    "unique_patients": [
        groups_train.nunique(),
        groups_test.nunique()
    ],
    "readmissions": [
        int(y_train.sum()),
        int(y_test.sum())
    ],
    "readmission_rate_pct": [
        round(y_train.mean() * 100, 2),
        round(y_test.mean() * 100, 2)
    ]
})

split_summary.to_csv(
    tables_path / "patient_level_split_summary.csv",
    index=False
)

print("Shared patients:", len(shared_patients))
split_summary

Shared patients: 0


,split,encounters,unique_patients,readmissions,readmission_rate_pct
0,Training,79567,55989,9070,11.40
1,Testing,19773,13998,2244,11.35


In [6]:
numeric_features = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses"
]

categorical_features = [
    column
    for column in feature_columns
    if column not in numeric_features
]

numeric_pipeline = Pipeline(steps=[
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    )
])

categorical_pipeline = Pipeline(steps=[
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value="Missing"
        )
    ),
    (
        "one_hot",
        OneHotEncoder(
            handle_unknown="ignore"
        )
    )
])

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Numeric features: 8
Categorical features: 36


## Split validation

The patient-level split produced no overlap between training and testing patients. The 30-day readmission rate was 11.40% in training data and 11.35% in testing data, indicating that the outcome distribution remained well balanced across the two patient groups.

All preprocessing will be embedded inside model pipelines fitted only on the training set. The test set will remain untouched until final model evaluation.

In [7]:
from time import perf_counter

import joblib

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

In [9]:
dummy_pipeline = Pipeline(steps=[
    (
        "preprocessor",
        clone(preprocessor)
    ),
    (
        "model",
        DummyClassifier(
            strategy="prior"
        )
    )
])

logistic_pipeline = Pipeline(steps=[
    (
        "preprocessor",
        clone(preprocessor)
    ),
    (
        "model",
        LogisticRegression(
            max_iter=1000,
            solver="liblinear",
            random_state=42
        )
    )
])

random_forest_pipeline = Pipeline(steps=[
    (
        "preprocessor",
        clone(preprocessor)
    ),
    (
        "model",
        RandomForestClassifier(
            n_estimators=150,
            max_depth=12,
            min_samples_leaf=10,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=42
        )
    )
])

models = {
    "Dummy classifier": dummy_pipeline,
    "Logistic regression": logistic_pipeline,
    "Random forest": random_forest_pipeline
}

In [10]:
training_times = {}
trained_models = {}

for model_name, model_pipeline in models.items():
    print(f"Training {model_name}...")

    start_time = perf_counter()

    model_pipeline.fit(X_train, y_train)

    elapsed_time = perf_counter() - start_time

    training_times[model_name] = elapsed_time
    trained_models[model_name] = model_pipeline

    print(
        f"Completed {model_name} in "
        f"{elapsed_time:.2f} seconds.\n"
    )

Training Dummy classifier...
Completed Dummy classifier in 0.69 seconds.

Training Logistic regression...
Completed Logistic regression in 4.67 seconds.

Training Random forest...
Completed Random forest in 13.06 seconds.



In [11]:
training_summary = pd.DataFrame({
    "model": list(training_times.keys()),
    "training_time_seconds": [
        round(value, 2)
        for value in training_times.values()
    ]
})

training_summary.to_csv(
    tables_path / "model_training_times.csv",
    index=False
)

training_summary

,model,training_time_seconds
0,Dummy classifier,0.69
1,Logistic regression,4.67
2,Random forest,13.06


In [12]:
models_path = project_root / "models"
models_path.mkdir(parents=True, exist_ok=True)

model_filenames = {
    "Dummy classifier": "dummy_classifier.joblib",
    "Logistic regression": "logistic_regression.joblib",
    "Random forest": "random_forest.joblib"
}

for model_name, filename in model_filenames.items():
    output_path = models_path / filename

    joblib.dump(
        trained_models[model_name],
        output_path
    )

    print(f"Saved {model_name}: {output_path}")

Saved Dummy classifier: c:\Users\ezont\Desktop\PersonalProj\hospital_readmission_prediction\models\dummy_classifier.joblib
Saved Logistic regression: c:\Users\ezont\Desktop\PersonalProj\hospital_readmission_prediction\models\logistic_regression.joblib
Saved Random forest: c:\Users\ezont\Desktop\PersonalProj\hospital_readmission_prediction\models\random_forest.joblib
